# Put a kernel on the FPGA's accelerator, with an LLM

Your board is a **PYNQ-Z1 FPGA** running a RISC-V core (Rocket, 40 MHz) with a small accelerator built in:
**MBP**, four instructions that each work on **eight int8 values at once**. In this notebook:

1. you meet **MBP.MAX8** and do its job by hand,
2. an LLM (DeepSeek, on AWS Bedrock) rewrites a max-pool kernel to use it, **with your FPGA in the loop**,
3. you find out how much of the speedup is the accelerator itself, and
4. you write the kernel yourself and race the LLM.

Run cells with **Shift-Enter**. How it works, in depth: `walkthroughs/` (left, in the file browser).

> Stuck, or something not working? **`mb_lab_solved.ipynb`** shows a complete run on a real board.

In [ ]:
import os, sys
for d in (os.getcwd(), os.path.expanduser("~/iiswc-tutorial/notebooks/mb_lab")):
    if os.path.exists(os.path.join(d, "mb_lab.py")):
        sys.path.insert(0, d)
import mb_lab as lab
lab.doctor()        # your seat, your board through its tunnel, and the LLM key

*Expected:* a checklist of green `ok`s and **READY**, then a card for your board (its name, address, WiFi, `FPGA operating`). A red line names what is wrong and what to do; `go()` still works without a board (it runs on spike only) or without the LLM (it replays a verified kernel), and says so.

## 1 · MBP.MAX8, by hand

A 2×2 max pool outputs the largest of four neighbouring bytes. The plain C kernel does that one byte at a
time: **118 cycles per output** on the board. MBP.MAX8 compares **eight pairs of bytes in one instruction**.
Here is the trick the fast kernel uses, on random bytes: two rows in, four outputs out, two instructions.

In [ ]:
lab.accelerator()   # run it again for other bytes

**✏️ Your turn.** `lab.max8(a, b)` is MBP.MAX8 in Python: eight lanes, the maximum of each pair. Pick two rows
of eight int8 values (from -128 to 127), **predict** the answer, then run the cell.

In [ ]:
row0 = [ ... ]            # ✏️ eight numbers between -128 and 127
row1 = [ ... ]            # ✏️ eight more
my_prediction = [ ... ]   # ✏️ what will MAX8 give?
print("MBP.MAX8 says:", lab.max8(row0, row1), "  you said:", my_prediction, "  ",
      "✓" if lab.max8(row0, row1) == my_prediction else "✗")

## 2 · The LLM rewrites the kernel, with your FPGA in the loop (5–8 minutes)

Each round, the LLM writes a few kernels. **Spike**, a simulator, checks each one and times it in seconds.
Then the best one of the round is built for your board and **runs on your FPGA**, and the LLM is told the
real cycle count for its next round. Blue bars are spike, orange bars are your FPGA, and the dashed outline
is the same kernel with the accelerator switched off.

**✏️ First, guess:** how many times faster will the LLM's kernel be on your FPGA?

In [ ]:
my_guess = ...    # ✏️ your guess: 2? 10? 50?

In [ ]:
lab.go("maxpool2d_s8")    # no LLM available? lab.go("maxpool2d_s8", "--replay") replays a verified kernel (2 min)

*Expected:* a chart that grows as the LLM tries kernels, then `done`. In 8 test runs it was 3.6–19.7× faster on the FPGA; the LLM writes a different kernel each time, so if yours is slow, run it again.

## 3 · Where the speedup came from

The verdict splits it in two: what the **rewritten loop** bought (the same kernel with MBP off, vs the
reference), and what the **accelerator** bought (the same kernel with MBP on, vs off). Spike's number is
higher than the board's. Spike charges one cycle per instruction and knows nothing about memory, and once
MAX8 makes the compute eight times denser, memory is what's left.

In [ ]:
lab.verdict()
print(f"your guess: {my_guess}x")

In [ ]:
lab.kernels()       # before and after; the highlighted lines are the accelerator

In [ ]:
lab.calls()         # every prompt the LLM got and every answer it gave: open the cards

### The tools, and every command the run executed

The lab is the usual tools, run in order:
- **ModelBlaster** turns the PyTorch model into an int8 graph and generates C kernels. With `--backend llm`
  it asks the LLM.
- **Zephyr's `west`** builds the images.
- **spike** simulates them.
- The **board's agent** runs them on the FPGA.

Here is every command the run executed, exactly. Paste any of them into a terminal.

In [ ]:
lab.commands()

In [ ]:
lab.sh("cd $ZCS && python -m modelblaster.pipeline.generate_kernels --help | head -40")   # ModelBlaster's own CLI

## 4 · Write the kernel yourself (≈2 minutes per try)

`lab.start()` puts the unoptimized kernel in **`your-kernel/maxpool2d_s8.c`** (left, in the file browser). Its header has
the rules and four hints: read one at a time. Edit, save with **Ctrl-S**, and run `lab.try_kernel()`. It checks
your kernel on spike (a wrong one never reaches the board), then runs it on your FPGA three ways.
**Goal: `ON THE ACCELERATOR` and under 10 cycles per output.**

In [ ]:
lab.start()

In [ ]:
lab.try_kernel()    # run it again after every edit; your scoreboard is below it

*Expected:* the reference and your kernel on spike and on your FPGA, the verdict, and your scoreboard. A wrong kernel stops on spike with the reason; a kernel that does not compile shows the compiler's error.

In [ ]:
# lab.solution()    # stuck? un-comment to see a solution (15.3x on the FPGA)

## 5 · More to try

* `lab.go("maxpool2d_s8", "--guide", "modelblaster")`: the LLM is **not** told about the accelerator. Does it find it?
* `lab.go("linear_s8")`: an int8 matrix multiply on MBP.DOT8, eight multiply-adds per instruction.
* `lab.go("gelu_s8")`: 43–49× faster with **no** accelerator at all. What did the LLM do instead?
* `lab.runs()` lists every run on this seat. `lab.verdict("<run>")`, `lab.kernels("<run>")`,
  `lab.calls("<run>")` and `lab.commands("<run>")` open one again.
* The same lab from a terminal (File → New → Terminal): `mb doctor`, `mb`, `mb try maxpool2d_s8`.